In [ ]:
import os, sys
cur_directory = os.getcwd()
project_folder = os.path.abspath(os.path.join(cur_directory, os.pardir, os.pardir))
save_directory = os.path.join(cur_directory, 'saved')
sys.path.append(os.path.join(project_folder, 'analysis-suite'))

In [ ]:
import numpy as np
from numpy import linalg as la
import matplotlib as mpl
from matplotlib import pyplot as plt
import analyses as als
import datasets as ds
import models as mds
import tools
from random import sample
from tqdm import tqdm
from argparse import Namespace
from time import gmtime, strftime
import pickle
import pickle

# Run

In [ ]:
BUSHES = 0
WATER = 1
LEAVES = 2
FISH = 3
OPTICFLOW = 4
stim_names = ['bushes', 'water', 'leaves', 'fish', 'opticflow']

def get_dataset(mov_ind):
    data_path = os.path.join(project_folder, 'data', 'nat_scenes_data')
    A = ds.DataNatScenes(data_path, mov_ind=mov_ind)
    return A

In [ ]:
V_width = 1
V_E_buffer = 0
E_width = 1
N_meta_neurons = 10

N_training_repeats = 10
critic_params = {
    "hidden_dim": 32,
    "embed_dim": 30,
    "layers": 2,
    "activation": "tanh",
}
opt_params = {"track_quantities": False, "quiet": True, "iterations": 16}
mi_measurement_params = {"quiet": True, "batch_size": 500, "N_batches": 500}

train_fraction = 0.5
samp_keep = 1

# shuffle time bins in a way that is the same across movie repeats
shuffle_time = False

# Put variable names here for parameters we want saved.
save_vars = {
    "N_meta_neurons",
    "mi_measurement_params",
    "N_training_repeats",
    "critic_params",
    "opt_params",
    "V_width",
    "V_E_buffer",
    "E_width",
    "train_fraction",
    "samp_keep",
    "shuffle_time"
}

In [ ]:
def test_cg_on_dataset(coarse_grainer, dataset, points=None):
    # setup experiment
    A = dataset
    A.reset()
    A.trim(20)
    V, E = A.create_V_and_E_datasets(V_width, V_E_buffer, E_width)
    
    # initialize critic function
    critic_function = mds.SeparableCritic(**critic_params)
    
    # Train MI estimator
    T = coarse_grainer(V.X.T).numpy().T
    als.train_NCE(X=T, Y=E.X, critic_function=critic_function, **opt_params)

    if points is None:
        train_points, test_points = tools.generate_train_test_indices(V.P, train_fraction)
    else:
        train_points, test_points = points

    # estimate MI for training points
    V.subset_points(train_points)
    E.subset_points(train_points)
    T = coarse_grainer(V.X.T).numpy().T
    mi_train = als.MI_NCE(
        X=T, Y=E.X, critic_function=critic_function, **mi_measurement_params
    )

    # test encoding i at time j for testing points
    V.reset()
    E.reset()
    V.subset_points(test_points)
    E.subset_points(test_points)
    T = coarse_grainer(V.X.T).numpy().T
    mi_test = als.MI_NCE(
        X=T, Y=E.X, critic_function=critic_function, **mi_measurement_params
    )
    
    return mi_train, mi_test

In [ ]:
def train_cg_on_dataset(dataset):
    A = dataset
    A.reset()
    A.trim(20)
    V, E = A.create_V_and_E_datasets(V_width, V_E_buffer, E_width, samp_keep)
    train_points, test_points = tools.generate_train_test_indices(V.P, train_fraction)
    
    # initialize models
    coarse_grainer = mds.PolynomialLayer(output_dim=N_meta_neurons, order=1)
    critic_function = mds.SeparableCritic(**critic_params)
    
    # Train MI estimator
    als.IB(X=V.X, Y=E.X, coarse_grainer=coarse_grainer, critic_function=critic_function, **opt_params)
    
    # return coarse_grainer, critic_function, (train_points, test_points)
    return coarse_grainer, (train_points, test_points)
    

In [ ]:
# First, compute coarse-graining maps

experiment_dims = (5, N_training_repeats)
coarse_grainers = np.zeros(experiment_dims, dtype=object)
train_test_points = np.zeros(experiment_dims, dtype=object)

for stim in tqdm(range(5)):
    for n in range(N_training_repeats):
        D = get_dataset(stim)
        coarse_grainers[stim, n], train_test_points[stim, n] = train_cg_on_dataset(D)

In [ ]:
# Now compute performance of each coarse-grainer on each dataset.

experiment_dims = (5, 5, N_training_repeats)
mi_train = np.zeros(experiment_dims, dtype=object)
mi_test = np.zeros(experiment_dims, dtype=object)

for stim_1 in tqdm(range(5)):
    for stim_2 in range(5):
        for n in range(N_training_repeats):
            cur_inds = (stim_1, stim_2, n)
            cg = coarse_grainers[stim_1, n]
            D = get_dataset(stim_2)
            points = train_test_points[stim_2, n]
            mi_train[cur_inds], mi_test[cur_inds] = test_cg_on_dataset(cg, D, points)

In [ ]:
# Now compute performance of each coarse-grainer on each dataset.

experiment_dims = (5, 5)
mi_average_train = np.zeros(experiment_dims)
mi_average_test = np.zeros(experiment_dims)
mi_stdevs_train = np.zeros(experiment_dims)
mi_stdevs_test = np.zeros(experiment_dims)

for stim_1 in tqdm(range(5)):
    for stim_2 in range(5):
        cur_inds = (stim_1, stim_2)
        cur_mi_train = np.array([a["mi"] for a in mi_train[cur_inds]])
        cur_mi_test = np.array([a["mi"] for a in mi_test[cur_inds]])
        # cur_mi_stdev_train = np.array([a["mi_stdev"] for a in mi_train[cur_inds]])
        # cur_mi_stdev_test = np.array([a["mi_stdev"] for a in mi_test[cur_inds]])
        
        mi_average_train[cur_inds] = np.mean(cur_mi_train)
        mi_average_test[cur_inds] = np.mean(cur_mi_test)
        mi_stdevs_train[cur_inds] = np.sqrt(np.var(cur_mi_train))
        mi_stdevs_test[cur_inds] = np.sqrt(np.var(cur_mi_test))

# Save

In [ ]:
save_vars = {'coarse_grainers', 'train_test_points', 'mi_test', 'mi_train', 'mi_average_train', 'mi_average_test', 'mi_stdevs_train', 'mi_stdevs_test'}.union(save_vars)
local_dict = locals()
output_data = {k:local_dict[k] for k in save_vars}

In [ ]:
timestamp = strftime("%Y.%m.%d %Hh %Mm %Ss.dat",gmtime())
# full_file_path = os.path.join(save_directory, '', strftime("%Y.%m.%d %Hh %Mm %Ss",gmtime())+'.dat')
buffer_suffix = f'_{V_E_buffer}_buffer' if V_E_buffer != 0 else ''
full_file_path = os.path.join(save_directory, f'pairwise_{N_training_repeats}_repeats_{opt_params["iterations"]}_iter{buffer_suffix}.dat')
# full_file_path = os.path.join(save_directory, 'test.dat')
os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
with open(full_file_path, 'wb+') as f:
            pickle.dump(output_data, f)
tools.record_parameters(full_file_path, output_data)

# Load

In [ ]:
load_path = './saved/pairwise_10_repeats_16_iter.dat'
# load_path = project_folder + '/analyses/cross-stim performance/saved/pairwise_10_repeats_16_iter.dat'
# load_path = project_folder + 'analyses/IB curves/saved/test.dat'
with open(load_path, 'rb') as f:
    data_dict = pickle.load(f)
data = Namespace(**data_dict)

In [ ]:
y_normed = data.mi_average_test / np.diag(data.mi_average_test)[None,:]
y = data.mi_average_test / np.log(2)

In [ ]:
plt.imshow(y_normed, vmin=0, vmax=1)
# plt.imshow(y, vmin=0)
plt.colorbar()
plt.xlabel('Tested on')
plt.ylabel('Trained on')

In [ ]:
plt.plot(y_normed)

In [ ]:
y

In [ ]:
y_ctl = np.diag(y)
y_nov = np.sum((np.ones_like(y) - np.eye(5)) * y, axis=0) / 4